In [ ]:
import os
import torch
import rlcard
from rlcard.agents.dmc_agent import DMCAgent
from rlcard.envs.registration import register
from envs.tienlen_env import TienLenEnv
from rlcard.agents import RandomAgent
from rlcard.utils import tournament


In [ ]:
register(env_id='tien-len-v1', entry_point=TienLenEnv)
env = rlcard.make('tien-len-v1')


In [ ]:
agents = []
for i in range(env.num_players):
    agent = DMCAgent(
        state_shape=env.state_shape[i],
        action_shape=[52], # The 52-bit bitmap feature vector
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )
    agents.append(agent)

env.set_agents(agents)


In [1]:
def train(num_episodes=10000):
    print("Starting Training...")
    for episode in range(num_episodes):
        # env.run() collects a full game trajectory using self-play
        trajectories, payoffs = env.run(is_training=True)

        # Feed the data into the agents for learning
        for i in range(env.num_players):
            for ts in trajectories[i]:
                # DMC agents update their internal networks here
                agents[i].feed(ts, payoffs[i])

        if episode % 100 == 0:
            print(f"Episode {episode} completed. Winner Payoff: {max(payoffs)}")
            # Save the first agent as our "Main" model
            agents[0].save(os.path.join('models', f'tienlen_model_{episode}.pth'))

def evaluate():
    # 1. Load the trained agent
    trained_agent = DMCAgent(state_shape=[159], action_shape=[52])
    trained_agent.load('models/tienlen_model_final.pth')

    # 2. Create naive opponents
    random_agent = RandomAgent(num_actions=env.num_actions)

    # 3. Set up the environment for evaluation
    # Agent 0 is our model, Agents 1-3 are random
    env.set_agents([trained_agent, random_agent, random_agent, random_agent])

    # 4. Run Tournament
    print("Running Tournament...")
    rewards = tournament(env, 1000) # Play 1000 games

    print(f"Average Rewards: {rewards}")
    print(f"Trained Agent Win Rate: {rewards[0]}")

def play_test_game():
    state, player_id = env.reset()
    while not env.is_over():
        agent = agents[player_id]
        # eval_step disables exploration (chooses only the highest-scored move)
        action, info = agent.eval_step(state)

        print(f"Player {player_id} plays: {action}")
        state, player_id = env.step(action)

    print("Game Over! Payoffs:", env.get_payoffs())


In [3]:
if not os.path.exists('models'): os.makedirs('models')
train()

In [ ]:
evaluate()

In [ ]:
play_test_game()